In [ ]:
import asyncio
import hashlib
import json
import os
import sys
import time
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv


In [ ]:
EXP_DIR = Path.cwd()
PROMPTS_DIR = EXP_DIR / "prompts"
REPO_ROOT = EXP_DIR.parents[1]

for p in (str(EXP_DIR), str(REPO_ROOT)):
    if p not in sys.path:
        sys.path.insert(0, p)

load_dotenv(REPO_ROOT / ".env")

import arbiter_lib as arb
from nli_lib.constants import (
    ARBITER_DIR,
    DISAGREEMENTS_CSV,
    adjudicated_csv,
    arbiter_cache_jsonl,
    arbiter_stats_json,
)

ARBITER_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
assert os.environ.get("OPENAI_API_KEY"), "OPENAI_API_KEY chưa có trong env / .env"
assert DISAGREEMENTS_CSV.exists(), f"Chạy pairing trước để có {DISAGREEMENTS_CSV}"

# Đổi v1 / v2 / v3 để chạy từng prompt arbiter rồi Run All.
VERSION = "v1"
print(f"VERSION = {VERSION}")


## 1. Tạo full prompt cho 1 disagreement row


In [ ]:
def load_prompt(version: str) -> tuple[str, str]:
    raw = (PROMPTS_DIR / f"arbiter_{version}.txt").read_text(encoding="utf-8")
    if "### SYSTEM" not in raw or "### STATUS DEFINITIONS" not in raw:
        raise ValueError(f"prompts/arbiter_{version}.txt thiếu '### SYSTEM' hoặc '### STATUS DEFINITIONS'")
    _, _, after_sys = raw.partition("### SYSTEM")
    sys_part, _, status_part = after_sys.partition("### STATUS DEFINITIONS")
    # Normalize CRLF -> LF để SHA256 prompt ổn định giữa các OS.
    status_block = "## STATUS DEFINITIONS" + status_part.replace("\r\n", "\n").rstrip()
    return sys_part.strip(), status_block

system_msg, status_block = load_prompt(VERSION)
print(f"system_msg: {len(system_msg)} chars; status_block: {len(status_block)} chars")


In [ ]:
def render_prompt(req: dict, status_block: str) -> str:
    if req["evidence_chunks"]:
        ev_lines = []
        for c in req["evidence_chunks"]:
            page = c.get("page_start")
            section = c.get("section_label") or ""
            header = f"[{c['chunk_id']}"
            if page is not None:
                header += f" | p.{page}"
            if section:
                header += f" | {section}"
            header += "]"
            ev_lines.append(f"{header}\n{c['content_text']}")
        evidence_block = "\n\n".join(ev_lines)
    else:
        evidence_block = (
            "(No chunks were cited by any system. The auditor must judge based "
            "on the requirement text alone — typically this means no_evidence.)"
        )

    sys_lines = []
    for sys_no in (1, 2, 3, 4):
        v = req["anon_verdicts"][sys_no - 1]
        cites = ", ".join(v.get("citations", [])) or "(none)"
        rationale = arb._truncate(v.get("rationale", ""), arb.RATIONALE_MAX_CHARS)
        sys_lines.append(
            f"System {sys_no}: status={v['status']}; "
            f"decision_path={v.get('decision_path', '')}; "
            f"citations={cites}\n  rationale: {rationale}"
        )
    systems_block = "\n".join(sys_lines)

    mt = req["material_topic"]
    mt_str = mt if mt != "__none__" else "(non-topical / general)"

    return f"""{status_block}

## REQUIREMENT
- Standard: {req["standard_id"]}
- Disclosure: {req["disclosure_id"]} — {req["disclosure_name"]}
- Material topic: {mt_str}
- Requirement-id: {req["requirement_id"]}
- Requirement text: {req["requirement_text"]}

## EVIDENCE CHUNKS (union of citations from all 4 systems)
{evidence_block}

## ANONYMIZED SYSTEM VERDICTS
{systems_block}

## TASK
Read all four verdicts and the evidence carefully, then decide:
1. correct_status: one of pass / partial / no_evidence — your authoritative call.
2. systems_correct: list of system numbers (subset of [1,2,3,4]) whose status equals your correct_status. Use [] if none agree.
3. rationale: 2-4 sentences explaining your reasoning, citing chunk_ids when possible.

Reply with ONLY a JSON object of this exact shape:
{{"correct_status": "...", "systems_correct": [1,2,3,4], "rationale": "..."}}
"""


## 3. Build request từ 1 disagreement row

Mỗi row của `disagreements.csv` chứa 4 verdict (a0/a1/a2/v_new). Convert thành 1 `request` dict gồm: requirement text + disclosure name (tra từ registry), permutation anonymize (a0/a1/a2/v_new → System 1/2/3/4 theo SHA256 của row identity), và evidence pack (union citation, cắt top 8 chunk theo tần suất).

Anonymize bằng `arbiter_lib.build_anon_layout`, evidence pack bằng `arbiter_lib.collect_evidence_chunks` — 2 helper shared với `run_human_sampling.py`.


In [ ]:
def row_to_request(row_idx: int, row: dict, req_map: dict, disc_map: dict) -> dict | None:
    standard_long = str(row["standard_id"])
    disc_id = str(row["disclosure_id"])
    req_id = str(row["requirement_id"])
    req_text = arb.lookup_requirement(req_map, standard_long, disc_id, req_id)
    if not req_text:
        print(f"  Missing requirement text for ({standard_long}, {disc_id}, {req_id}); skipping row {row_idx}")
        return None
    disc_name = arb.lookup_disclosure_name(disc_map, standard_long, disc_id)

    variant_to_system, system_to_variant = arb.build_anon_layout(row)

    anon_verdicts: list[dict] = [{}] * 4
    for variant in arb.PRIMARY_VARIANTS:
        sys_no = variant_to_system[variant]
        anon_verdicts[sys_no - 1] = {
            "status": str(row.get(f"{variant}_status", "")),
            "decision_path": str(row.get(f"{variant}_decision_path", "")),
            "rationale": str(row.get(f"{variant}_rationale", "")),
            "citations": arb._parse_citations(row.get(f"{variant}_citations_json")),
        }

    chunk_index = arb.load_chunks(str(row["report_id"]))
    evidence_chunks = arb.collect_evidence_chunks(row, chunk_index)

    return {
        "row_idx": row_idx,
        "report_id": str(row["report_id"]),
        "phase": str(row["phase"]),
        "standard_id": standard_long,
        "disclosure_id": disc_id,
        "material_topic": str(row.get("material_topic", "__none__")),
        "requirement_id": req_id,
        "occurrence_idx": int(row["occurrence_idx"]),
        "requirement_text": req_text,
        "disclosure_name": disc_name,
        "variant_to_system": variant_to_system,
        "system_to_variant": system_to_variant,
        "anon_verdicts": anon_verdicts,
        "evidence_chunks": evidence_chunks,
    }


def build_requests(df: pd.DataFrame, req_map: dict, disc_map: dict) -> list[dict]:
    requests: list[dict] = []
    for idx, row in df.iterrows():
        req = row_to_request(int(idx), row.to_dict(), req_map, disc_map)
        if req is not None:
            requests.append(req)
    return requests


## 4. Driver async: lặp render_prompt → adjudicate_one

`arbiter_lib.adjudicate_one` đóng gói toàn bộ phần "gọi LLM an toàn": kiểm tra cache (SHA256 prompt) → nếu miss thì gọi `call_with_retry` (exponential backoff + jitter cho transient error) → append cache JSONL → build result row.

Notebook chỉ lo orchestrate: tạo `AsyncOpenAI` client, `Semaphore` giới hạn concurrency, gọi worker song song bằng `asyncio.gather`, in tiến trình mỗi 50 request.


In [ ]:
async def adjudicate_all(requests: list[dict], cache_path: Path, system_msg: str, status_block: str):
    from openai import AsyncOpenAI

    cache = arb.load_cache(cache_path)
    print(f"Loaded {len(cache)} cache entries")

    counters = {"cache_hits": 0, "live_calls": 0, "errors": 0}
    client = AsyncOpenAI()
    sem = asyncio.Semaphore(arb.CONCURRENCY)

    async def runner(idx: int, req: dict) -> dict:
        prompt = render_prompt(req, status_block)
        out = await arb.adjudicate_one(req, prompt, client, cache, cache_path, system_msg, sem, counters)
        if (idx + 1) % 50 == 0 or (idx + 1) == len(requests):
            print(f"  progress {idx + 1}/{len(requests)}  hits={counters['cache_hits']}  live={counters['live_calls']}  errs={counters['errors']}")
        return out

    results = await asyncio.gather(*[runner(i, r) for i, r in enumerate(requests)])
    return list(results), counters


## 5. Load disagreements + chạy adjudicate

`material_topic` NaN → sentinel `__none__` để đảm bảo cache key ổn định giữa các lần chạy (NaN trong key sẽ làm hash khác nhau). Sau khi build requests, gọi driver async.


In [ ]:
df = pd.read_csv(DISAGREEMENTS_CSV)
df["material_topic"] = df["material_topic"].fillna("__none__")
print(f"{len(df)} disagreement rows")

req_map, disc_map = arb.load_registry()
requests = build_requests(df, req_map, disc_map)
print(f"{len(requests)} requests built (skipped {len(df) - len(requests)})")

cache_path = arbiter_cache_jsonl(VERSION)

t0 = time.time()
results, counters = await adjudicate_all(requests, cache_path, system_msg, status_block)
elapsed = time.time() - t0
print(f"Done in {elapsed:.1f}s; cache_hits={counters['cache_hits']} live_calls={counters['live_calls']} errors={counters['errors']}")


## 6. Lưu 

- `data/02_arbiter_postfix/adjudicated_{VERSION}.csv` — 1 row / disagreement


In [ ]:
OUTPUT_COLUMNS = [
    "report_id", "phase", "standard_id", "disclosure_id", "material_topic",
    "requirement_id", "occurrence_idx",
    "correct_status", "systems_correct_json",
    "a0_correct", "a1_correct", "a2_correct", "v_new_correct",
    "n_systems_correct", "arbiter_rationale",
    "n_evidence_chunks", "prompt_sha256", "cache_hit", "error",
]

out_df = pd.DataFrame(results, columns=OUTPUT_COLUMNS)
output_csv = adjudicated_csv(VERSION)
out_df.to_csv(output_csv, index=False, encoding="utf-8")
print(f"Wrote {len(out_df)} rows -> {output_csv.name}")

success = out_df[out_df["error"] == ""]
stats = {
    "n_disagreements_input": int(len(df)),
    "n_requests_built": int(len(requests)),
    "n_results": int(len(out_df)),
    "n_success": int(len(success)),
    "n_errors": int((out_df["error"] != "").sum()),
    "cache_hits": int(counters["cache_hits"]),
    "live_calls": int(counters["live_calls"]),
    "elapsed_seconds": round(elapsed, 2),
    "model": arb.MODEL,
    "prompt_version": VERSION,
    "n_results_with_unanimous_4": int((success["n_systems_correct"] == 4).sum()),
    "n_results_with_no_correct_system": int((success["n_systems_correct"] == 0).sum()),
    "per_variant_correct_count": {v: int(success[f"{v}_correct"].sum()) for v in arb.PRIMARY_VARIANTS},
    "per_variant_correct_rate": {
        v: round(float(success[f"{v}_correct"].mean()), 4) if len(success) else 0.0
        for v in arb.PRIMARY_VARIANTS
    },
}
stats_path = arbiter_stats_json(VERSION)
stats_path.write_text(json.dumps(stats, indent=2, ensure_ascii=False), encoding="utf-8")
print(f"Wrote stats -> {stats_path.name}")
print("Per-variant correct rate:", {k: f"{v:.3f}" for k, v in stats["per_variant_correct_rate"].items()})
